# Sistema Inteligente de Reslotting — Inchcape Perú
## MVP en Python: Machine Learning + Scoring multicriterio + Optimización

Este notebook construye un **MVP de decisión para reslotting** utilizando los datos disponibles del Centro de Distribución.

### Objetivo
Identificar qué SKU presentan mayor potencial de mejora y determinar una **zona recomendada factible** buscando reducir el tiempo estimado de picking.

### Flujo del notebook

1. Carga y exploración de datos.
2. Limpieza de las fuentes.
3. Construcción de indicadores por SKU.
4. Creación de una Base Maestra.
5. Cálculo de carga operativa y ahorro teórico.
6. Análisis ABC.
7. Score multicriterio.
8. Evaluación de las 900 combinaciones SKU × Zona.
9. Preparación de capacidades.
10. Configuración y construcción del optimizador.
11. Ejecución del modelo.
12. Extracción de recomendaciones.
13. Cálculo de KPI.
14. Validación de factibilidad.
15. Visualización de resultados.
16. Exportación a Excel.
17. Salida ejecutiva y conclusiones.
18. Preparación del dataset de Machine Learning.
19. Escalado de variables.
20. Selección del número de clusters.
21. Entrenamiento de K-Means.
22. Evaluación del modelo ML.
23. Interpretación de clusters.
24. Integración del ML con el motor de decisión.
25. Resultado híbrido ML + Optimización.
26. Visualización del modelo ML.
27. Exportación del modelo y resultados finales.

> **Importante:** con el dataset actual no existe una serie temporal con fechas ni una variable objetivo de “zona óptima”. Por ello, el componente de Machine Learning de este notebook es **no supervisado** y utiliza **K-Means** para descubrir perfiles de SKU. La ubicación recomendada continúa siendo responsabilidad del optimizador matemático, que respeta las restricciones disponibles. Cuando exista histórico temporal podrá añadirse un modelo supervisado de forecast.

# Fase 0 — Preparación del entorno

## 0.1 Instalación de librerías

Utilizaremos:

- **Pandas:** manipulación y análisis de datos.
- **NumPy:** cálculos numéricos.
- **OpenPyXL:** lectura y generación de archivos Excel.
- **PuLP:** optimización matemática.
- **Matplotlib:** visualización de resultados.

Google Colab ya incluye varias de estas librerías, pero ejecutamos la instalación para asegurar que el entorno esté preparado.

In [ ]:
!pip -q install pandas numpy openpyxl pulp matplotlib

## 0.2 Importación de librerías

Importamos las herramientas que utilizaremos durante todo el proyecto y configuramos Pandas para facilitar la visualización de tablas.

In [ ]:
import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pulp

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("✅ Librerías importadas correctamente.")

✅ Librerías importadas correctamente.


# Fase 1 — Carga y exploración del dataset

## 1.1 Subir el archivo Excel

Esta celda permite cargar el archivo directamente desde la computadora hacia Google Colab.

Selecciona el Excel original de Inchcape cuando aparezca el botón para subir archivos.

In [ ]:
from google.colab import files

uploaded = files.upload()

ARCHIVO = list(uploaded.keys())[0]

print("✅ Archivo cargado:")
print(ARCHIVO)

## 1.2 Identificación de hojas disponibles

Antes de leer los datos verificamos los nombres exactos de las hojas existentes dentro del archivo.

In [ ]:
excel = pd.ExcelFile(ARCHIVO)

print("Hojas encontradas en el archivo:")
for hoja in excel.sheet_names:
    print("-", hoja)

## 1.3 Carga de las hojas fuente

El archivo contiene hojas de entrada y también hojas con análisis derivados.

Para construir el modelo utilizaremos únicamente las fuentes originales:

- `MAESTRO_SKUs`
- `ROTACIÓN`
- `STOCK_ACTUAL`
- `LAYOUT_CD`
- `OCUPACION_POR_ZONA`
- `PEDIDOS ACTUAL`

Las hojas de dashboard o analytics no se utilizarán como entrada del modelo.

In [ ]:
maestro = pd.read_excel(ARCHIVO, sheet_name="MAESTRO_SKUs")
rotacion = pd.read_excel(ARCHIVO, sheet_name="ROTACIÓN")
stock = pd.read_excel(ARCHIVO, sheet_name="STOCK_ACTUAL")
layout = pd.read_excel(ARCHIVO, sheet_name="LAYOUT_CD")
ocupacion = pd.read_excel(ARCHIVO, sheet_name="OCUPACION_POR_ZONA")
pedidos = pd.read_excel(ARCHIVO, sheet_name="PEDIDOS ACTUAL")

print("✅ Hojas fuente cargadas correctamente.")

## 1.4 Dimensiones del dataset

Verificamos cuántos registros y columnas contiene cada fuente.

In [ ]:
datasets = {
    "MAESTRO_SKUs": maestro,
    "ROTACIÓN": rotacion,
    "STOCK_ACTUAL": stock,
    "LAYOUT_CD": layout,
    "OCUPACION_POR_ZONA": ocupacion,
    "PEDIDOS ACTUAL": pedidos
}

print("DIMENSIONES DEL DATASET")
print("-" * 50)

for nombre, df in datasets.items():
    print(f"{nombre}: {df.shape}")

## 1.5 Nombres de las variables

Revisamos las columnas reales para detectar campos útiles y columnas auxiliares generadas en Excel.

In [ ]:
for nombre, df in datasets.items():
    print("\n" + "=" * 60)
    print(nombre)
    print("=" * 60)
    print(df.columns.tolist())

# Fase 2 — Limpieza y preparación de datos

## 2.1 Selección de variables necesarias

Algunas hojas contienen columnas auxiliares como `Unnamed`.

Seleccionamos únicamente las variables necesarias para el modelo de reslotting.

In [ ]:
maestro_limpio = maestro[
    ["SKU", "MARCA", "FAMILIA", "VOLUMEN_M3", "PESO_KG"]
].copy()

rotacion_limpio = rotacion[
    ["SKU", "ROTACION_6M", "ABC"]
].copy()

stock_limpio = stock[
    ["UBICACIÓN", "SKU", "ZONA_ACTUAL"]
].copy()

layout_limpio = layout[
    ["ZONA", "DISTANCIA_METROS", "TIEMPO_MINUTOS", "CAPACIDAD_M3_MAX"]
].copy()

ocupacion_limpio = ocupacion[
    [
        "ZONA",
        "CAPACIDAD_MAX_M3",
        "VOLUMEN_USADO_M3",
        "VOLUMEN_DISPONIBLE_M3",
        "PORCENTAJE_USO_%"
    ]
].copy()

pedidos_limpio = pedidos[
    ["PEDIDO_ID", "LINEA", "SKU", "CANTIDAD", "ZONA_ACTUAL", "TIEMPO_HOY_MIN"]
].copy()

print("✅ Columnas necesarias seleccionadas.")

## 2.2 Validación de las tablas limpias

Comprobamos nuevamente sus dimensiones y columnas.

In [ ]:
datasets_limpios = {
    "MAESTRO": maestro_limpio,
    "ROTACIÓN": rotacion_limpio,
    "STOCK": stock_limpio,
    "LAYOUT": layout_limpio,
    "OCUPACIÓN": ocupacion_limpio,
    "PEDIDOS": pedidos_limpio
}

for nombre, df in datasets_limpios.items():
    print("\n", nombre)
    print("Dimensión:", df.shape)
    print("Columnas:", df.columns.tolist())

## 2.3 Validación de SKU y estructura

El Maestro, Rotación y Stock deberían contener los mismos 100 SKU. También verificamos pedidos y zonas.

In [ ]:
print("VALIDACIÓN DE SKU")
print("-" * 45)

print("SKU únicos en Maestro:", maestro_limpio["SKU"].nunique())
print("SKU únicos en Rotación:", rotacion_limpio["SKU"].nunique())
print("SKU únicos en Stock:", stock_limpio["SKU"].nunique())
print("SKU únicos en Pedidos:", pedidos_limpio["SKU"].nunique())
print("Pedidos diferentes:", pedidos_limpio["PEDIDO_ID"].nunique())
print("Líneas de pedido:", len(pedidos_limpio))
print("Zonas disponibles:", layout_limpio["ZONA"].nunique())

## 2.4 Revisión de valores faltantes y duplicados

Antes de integrar las fuentes comprobamos faltantes y duplicados en identificadores clave.

In [ ]:
for nombre, df in datasets_limpios.items():
    faltantes = int(df.isna().sum().sum())
    print(f"{nombre}: {faltantes} valores faltantes totales")

print("\nDuplicados por SKU:")
print("Maestro:", maestro_limpio["SKU"].duplicated().sum())
print("Rotación:", rotacion_limpio["SKU"].duplicated().sum())
print("Stock:", stock_limpio["SKU"].duplicated().sum())

# Fase 3 — Construcción de indicadores por SKU

## 3.1 Agregación de pedidos

La hoja `PEDIDOS ACTUAL` contiene 1,500 líneas. El modelo necesita trabajar a nivel de SKU.

Calcularemos:

- **N_LINEAS:** apariciones del SKU en las líneas de pedido.
- **N_PEDIDOS:** pedidos distintos en los que aparece.
- **CANT_TOTAL:** unidades totales solicitadas.
- **CANT_PROMEDIO:** cantidad promedio por línea.
- **TIEMPO_OBSERVADO_PROM:** tiempo promedio registrado.
- **TIEMPO_OBSERVADO_TOTAL:** tiempo acumulado observado.

`N_LINEAS` se utilizará como aproximación a la frecuencia de visitas a la ubicación.

In [ ]:
pedidos_por_sku = (
    pedidos_limpio
    .groupby("SKU")
    .agg(
        N_LINEAS=("SKU", "size"),
        N_PEDIDOS=("PEDIDO_ID", "nunique"),
        CANT_TOTAL=("CANTIDAD", "sum"),
        CANT_PROMEDIO=("CANTIDAD", "mean"),
        TIEMPO_OBSERVADO_PROM=("TIEMPO_HOY_MIN", "mean"),
        TIEMPO_OBSERVADO_TOTAL=("TIEMPO_HOY_MIN", "sum")
    )
    .reset_index()
)

print("✅ Tabla de pedidos por SKU creada.")
print("Dimensión:", pedidos_por_sku.shape)

pedidos_por_sku.head(10)

## 3.2 Variables derivadas de demanda

Creamos `UNIDADES_POR_PEDIDO` para diferenciar productos solicitados muchas veces en pequeñas cantidades frente a productos solicitados pocas veces en lotes grandes.

In [ ]:
pedidos_por_sku["UNIDADES_POR_PEDIDO"] = np.where(
    pedidos_por_sku["N_PEDIDOS"] > 0,
    pedidos_por_sku["CANT_TOTAL"] / pedidos_por_sku["N_PEDIDOS"],
    0
)

pedidos_por_sku[
    ["SKU", "N_LINEAS", "N_PEDIDOS", "CANT_TOTAL", "CANT_PROMEDIO", "UNIDADES_POR_PEDIDO"]
].head(10)

## 3.3 SKU con mayor frecuencia observada

Revisamos los productos que aparecen más veces en las líneas de pedido.

In [ ]:
top_frecuencia = (
    pedidos_por_sku
    .sort_values("N_LINEAS", ascending=False)
    .head(15)
)

top_frecuencia[
    ["SKU", "N_LINEAS", "N_PEDIDOS", "CANT_TOTAL", "UNIDADES_POR_PEDIDO"]
]

## 3.4 SKU con mayor cantidad total solicitada

La frecuencia de visitas y la cantidad total de unidades no representan exactamente lo mismo, por lo que analizamos ambas.

In [ ]:
top_cantidad = (
    pedidos_por_sku
    .sort_values("CANT_TOTAL", ascending=False)
    .head(15)
)

top_cantidad[
    ["SKU", "N_LINEAS", "N_PEDIDOS", "CANT_TOTAL", "CANT_PROMEDIO", "UNIDADES_POR_PEDIDO"]
]

# Fase 4 — Construcción de la Base Maestra

## 4.1 Integración de Maestro + Rotación + Stock

Unimos las características físicas, la rotación y la ubicación actual utilizando `SKU` como identificador.

In [ ]:
base_maestra = (
    maestro_limpio
    .merge(rotacion_limpio, on="SKU", how="left")
    .merge(stock_limpio, on="SKU", how="left")
)

print("Base inicial construida.")
print("Dimensión:", base_maestra.shape)

base_maestra.head()

## 4.2 Incorporación de indicadores de pedidos

Añadimos la información agregada de los pedidos a la Base Maestra.

In [ ]:
base_maestra = base_maestra.merge(
    pedidos_por_sku,
    on="SKU",
    how="left"
)

columnas_pedido = [
    "N_LINEAS",
    "N_PEDIDOS",
    "CANT_TOTAL",
    "CANT_PROMEDIO",
    "TIEMPO_OBSERVADO_PROM",
    "TIEMPO_OBSERVADO_TOTAL",
    "UNIDADES_POR_PEDIDO"
]

base_maestra[columnas_pedido] = base_maestra[columnas_pedido].fillna(0)

print("✅ Información de pedidos incorporada.")
print("Dimensión:", base_maestra.shape)

## 4.3 Incorporación del layout de la zona actual

Cada SKU se relaciona con la distancia, el tiempo y la capacidad de su zona actual.

In [ ]:
layout_actual = layout_limpio.rename(
    columns={
        "ZONA": "ZONA_ACTUAL",
        "DISTANCIA_METROS": "DISTANCIA_ACTUAL_M",
        "TIEMPO_MINUTOS": "TIEMPO_LAYOUT_ACTUAL",
        "CAPACIDAD_M3_MAX": "CAPACIDAD_ZONA_ACTUAL"
    }
)

base_maestra = base_maestra.merge(
    layout_actual,
    on="ZONA_ACTUAL",
    how="left"
)

print("✅ Layout incorporado.")
print("Dimensión final:", base_maestra.shape)

## 4.4 Validación de la Base Maestra

Comprobamos que la integración mantenga 100 SKU sin duplicados ni campos críticos faltantes.

In [ ]:
print("=" * 55)
print("VALIDACIÓN BASE MAESTRA")
print("=" * 55)

print("Número de registros:", len(base_maestra))
print("SKU únicos:", base_maestra["SKU"].nunique())
print("SKU duplicados:", base_maestra["SKU"].duplicated().sum())
print("SKU sin zona:", base_maestra["ZONA_ACTUAL"].isna().sum())
print("SKU sin tiempo:", base_maestra["TIEMPO_LAYOUT_ACTUAL"].isna().sum())
print("SKU sin rotación:", base_maestra["ROTACION_6M"].isna().sum())

assert base_maestra["SKU"].nunique() == len(base_maestra)
assert base_maestra["ZONA_ACTUAL"].notna().all()
assert base_maestra["TIEMPO_LAYOUT_ACTUAL"].notna().all()

print("\n✅ Base Maestra validada.")

# Fase 5 — Medición del impacto operativo

## 5.1 Carga operativa actual

Definimos:

**Carga Operativa = Número de visitas × Tiempo de acceso actual**

Un SKU con alta frecuencia de visitas y ubicado en una zona lenta tendrá una carga mayor.

In [ ]:
base_maestra["CARGA_OPERATIVA_MIN"] = (
    base_maestra["N_LINEAS"] *
    base_maestra["TIEMPO_LAYOUT_ACTUAL"]
)

base_maestra[
    ["SKU", "ABC", "ZONA_ACTUAL", "N_LINEAS", "TIEMPO_LAYOUT_ACTUAL", "CARGA_OPERATIVA_MIN"]
].sort_values("CARGA_OPERATIVA_MIN", ascending=False).head(15)

## 5.2 Zona con menor tiempo de acceso

Identificamos la zona más rápida únicamente como referencia para calcular un **máximo teórico de ahorro**.

Esto no significa que todos los SKU puedan ser trasladados a esa zona.

In [ ]:
zona_mas_rapida = layout_limpio.sort_values("TIEMPO_MINUTOS").iloc[0]
tiempo_minimo_cd = layout_limpio["TIEMPO_MINUTOS"].min()

print("Zona con menor tiempo:", zona_mas_rapida["ZONA"])
print("Tiempo:", zona_mas_rapida["TIEMPO_MINUTOS"], "min")
print("Distancia:", zona_mas_rapida["DISTANCIA_METROS"], "m")

## 5.3 Ahorro máximo teórico

Calculamos:

**Ahorro teórico = N_LINEAS × (Tiempo actual - Tiempo mínimo)**

Sirve para medir el potencial de mejora antes de aplicar restricciones.

In [ ]:
base_maestra["AHORRO_TEORICO_MIN"] = (
    base_maestra["N_LINEAS"] *
    (base_maestra["TIEMPO_LAYOUT_ACTUAL"] - tiempo_minimo_cd)
).clip(lower=0)

base_maestra[
    [
        "SKU",
        "ZONA_ACTUAL",
        "N_LINEAS",
        "TIEMPO_LAYOUT_ACTUAL",
        "CARGA_OPERATIVA_MIN",
        "AHORRO_TEORICO_MIN"
    ]
].sort_values("AHORRO_TEORICO_MIN", ascending=False).head(20)

## 5.4 Ranking preliminar de impacto

Ordenamos los SKU según su ahorro máximo teórico.

In [ ]:
ranking_preliminar = (
    base_maestra
    .sort_values("AHORRO_TEORICO_MIN", ascending=False)
    .reset_index(drop=True)
)

ranking_preliminar["RANKING_PRELIMINAR"] = ranking_preliminar.index + 1

ranking_preliminar[
    [
        "RANKING_PRELIMINAR",
        "SKU",
        "MARCA",
        "FAMILIA",
        "ABC",
        "ROTACION_6M",
        "N_LINEAS",
        "N_PEDIDOS",
        "ZONA_ACTUAL",
        "CARGA_OPERATIVA_MIN",
        "AHORRO_TEORICO_MIN"
    ]
].head(20)

## 5.5 Identificación del 20% inicial

La hipótesis del MVP propone concentrar el esfuerzo aproximadamente en el 20% de SKU con mayor potencial de impacto.

Con una muestra de 100 SKU, el 20% equivale a 20 productos.

In [ ]:
cantidad_top = max(1, int(round(len(base_maestra) * 0.20)))

top_sku = ranking_preliminar.head(cantidad_top).copy()
top_sku["CANDIDATO_INICIAL"] = "SÍ"

print("Cantidad de SKU en el TOP 20%:", cantidad_top)

top_sku[
    [
        "RANKING_PRELIMINAR",
        "SKU",
        "ABC",
        "ROTACION_6M",
        "N_LINEAS",
        "ZONA_ACTUAL",
        "AHORRO_TEORICO_MIN",
        "CANDIDATO_INICIAL"
    ]
]

# Fase 6 — Análisis ABC vs. impacto operativo

## 6.1 Impacto según clasificación ABC

La clasificación ABC representa rotación, pero el reslotting es una decisión multivariable.

Analizamos si los productos A presentan necesariamente la mayor carga o el mayor ahorro.

In [ ]:
analisis_abc = (
    base_maestra
    .groupby("ABC")
    .agg(
        SKU=("SKU", "count"),
        ROTACION_PROMEDIO=("ROTACION_6M", "mean"),
        VISITAS_PROMEDIO=("N_LINEAS", "mean"),
        CARGA_PROMEDIO=("CARGA_OPERATIVA_MIN", "mean"),
        AHORRO_PROMEDIO=("AHORRO_TEORICO_MIN", "mean")
    )
    .reset_index()
)

analisis_abc

## 6.2 Distribución ABC dentro del TOP 20

Si aparecen SKU B o C entre los principales candidatos, esto evidencia que **ABC por sí solo no determina la ubicación óptima**.

In [ ]:
distribucion_top_abc = (
    top_sku["ABC"]
    .value_counts()
    .rename_axis("ABC")
    .reset_index(name="CANTIDAD_SKU")
)

distribucion_top_abc

# Fase 7 — Score Multicriterio

## 7.1 Normalización Min-Max

Las variables tienen escalas diferentes. Las convertimos a una escala de 0 a 1 antes de combinarlas.

In [ ]:
def normalizar_01(serie):
    serie = serie.astype(float)
    minimo = serie.min()
    maximo = serie.max()

    if pd.isna(minimo) or pd.isna(maximo) or maximo == minimo:
        return pd.Series(np.zeros(len(serie)), index=serie.index)

    return (serie - minimo) / (maximo - minimo)

print("✅ Función de normalización creada.")

## 7.2 Variables del Score

Utilizaremos inicialmente:

- Ahorro potencial.
- Rotación.
- Clasificación ABC.
- Facilidad relativa de movimiento basada en volumen.

La facilidad de movimiento es una aproximación provisional: menor volumen implica mayor facilidad relativa.

In [ ]:
base_maestra["AHORRO_NORM"] = normalizar_01(base_maestra["AHORRO_TEORICO_MIN"])
base_maestra["ROTACION_NORM"] = normalizar_01(base_maestra["ROTACION_6M"])
base_maestra["VOLUMEN_NORM"] = normalizar_01(base_maestra["VOLUMEN_M3"])

base_maestra["FACILIDAD_MOVIMIENTO"] = 1 - base_maestra["VOLUMEN_NORM"]

mapa_abc = {
    "A": 1.00,
    "B": 0.60,
    "C": 0.30
}

base_maestra["ABC_SCORE"] = base_maestra["ABC"].map(mapa_abc).fillna(0)

base_maestra[
    ["SKU", "ABC", "AHORRO_NORM", "ROTACION_NORM", "ABC_SCORE", "FACILIDAD_MOVIMIENTO"]
].head(10)

## 7.3 Pesos iniciales

Pesos configurables del MVP:

- 55% Ahorro potencial.
- 20% Rotación.
- 10% ABC.
- 15% Facilidad de movimiento.

Estos pesos deben validarse posteriormente con Operaciones.

In [ ]:
PESOS_SCORE = {
    "ahorro": 0.55,
    "rotacion": 0.20,
    "abc": 0.10,
    "facilidad_movimiento": 0.15
}

suma_pesos = sum(PESOS_SCORE.values())
assert abs(suma_pesos - 1) < 0.0001

print("✅ Pesos configurados correctamente. Suma =", suma_pesos)

## 7.4 Cálculo del Score de Prioridad

El Score se expresa en una escala de 0 a 100.

> El Score **prioriza** SKU; no decide por sí solo la zona final.

In [ ]:
base_maestra["SCORE_PRIORIDAD"] = 100 * (
    PESOS_SCORE["ahorro"] * base_maestra["AHORRO_NORM"]
    + PESOS_SCORE["rotacion"] * base_maestra["ROTACION_NORM"]
    + PESOS_SCORE["abc"] * base_maestra["ABC_SCORE"]
    + PESOS_SCORE["facilidad_movimiento"] * base_maestra["FACILIDAD_MOVIMIENTO"]
)

base_maestra["RANKING_SCORE"] = (
    base_maestra["SCORE_PRIORIDAD"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

ranking_score = (
    base_maestra
    .sort_values("SCORE_PRIORIDAD", ascending=False)
    .reset_index(drop=True)
)

ranking_score[
    [
        "RANKING_SCORE",
        "SKU",
        "MARCA",
        "FAMILIA",
        "ABC",
        "ROTACION_6M",
        "N_LINEAS",
        "ZONA_ACTUAL",
        "CARGA_OPERATIVA_MIN",
        "AHORRO_TEORICO_MIN",
        "SCORE_PRIORIDAD"
    ]
].head(20)

# Fase 8 — Evaluación SKU × Zona

## 8.1 Preparación de SKU y zonas

Evaluaremos cada SKU frente a todas las zonas disponibles.

In [ ]:
sku_para_evaluar = base_maestra[
    [
        "SKU",
        "MARCA",
        "FAMILIA",
        "ABC",
        "ROTACION_6M",
        "VOLUMEN_M3",
        "PESO_KG",
        "N_LINEAS",
        "N_PEDIDOS",
        "CANT_TOTAL",
        "ZONA_ACTUAL",
        "TIEMPO_LAYOUT_ACTUAL",
        "CARGA_OPERATIVA_MIN",
        "SCORE_PRIORIDAD",
        "RANKING_SCORE"
    ]
].copy()

zonas_para_evaluar = layout_limpio[
    ["ZONA", "DISTANCIA_METROS", "TIEMPO_MINUTOS", "CAPACIDAD_M3_MAX"]
].copy()

print("SKU a evaluar:", len(sku_para_evaluar))
print("Zonas disponibles:", len(zonas_para_evaluar))

## 8.2 Construcción de la matriz SKU × Zona

Con 100 SKU y 9 zonas se generan **900 escenarios**.

In [ ]:
sku_para_evaluar["KEY"] = 1
zonas_para_evaluar["KEY"] = 1

matriz_sku_zona = (
    sku_para_evaluar
    .merge(zonas_para_evaluar, on="KEY")
    .drop(columns="KEY")
)

print("Número total de escenarios:", len(matriz_sku_zona))
print("SKU únicos:", matriz_sku_zona["SKU"].nunique())
print("Zonas únicas:", matriz_sku_zona["ZONA"].nunique())

## 8.3 Costo operativo de cada escenario

Para cada combinación calculamos:

**Costo nuevo = N_LINEAS × Tiempo de la zona candidata**

**Ahorro = Costo actual - Costo nuevo**

In [ ]:
matriz_sku_zona["COSTO_NUEVO_MIN"] = (
    matriz_sku_zona["N_LINEAS"] *
    matriz_sku_zona["TIEMPO_MINUTOS"]
)

matriz_sku_zona["AHORRO_MIN"] = (
    matriz_sku_zona["CARGA_OPERATIVA_MIN"] -
    matriz_sku_zona["COSTO_NUEVO_MIN"]
)

matriz_sku_zona["ES_ZONA_ACTUAL"] = (
    matriz_sku_zona["ZONA"] == matriz_sku_zona["ZONA_ACTUAL"]
)

matriz_sku_zona[
    [
        "SKU",
        "ZONA_ACTUAL",
        "ZONA",
        "N_LINEAS",
        "TIEMPO_LAYOUT_ACTUAL",
        "TIEMPO_MINUTOS",
        "COSTO_NUEVO_MIN",
        "AHORRO_MIN",
        "ES_ZONA_ACTUAL"
    ]
].head(18)

## 8.4 Ejemplo de evaluación de un SKU

Seleccionamos el SKU con mayor Score y mostramos sus nueve alternativas.

In [ ]:
sku_ejemplo = ranking_score.iloc[0]["SKU"]

ejemplo_sku = (
    matriz_sku_zona[
        matriz_sku_zona["SKU"] == sku_ejemplo
    ]
    .sort_values("AHORRO_MIN", ascending=False)
)

print("SKU seleccionado:", sku_ejemplo)

ejemplo_sku[
    [
        "SKU",
        "ZONA_ACTUAL",
        "ZONA",
        "DISTANCIA_METROS",
        "TIEMPO_MINUTOS",
        "N_LINEAS",
        "COSTO_NUEVO_MIN",
        "AHORRO_MIN",
        "ES_ZONA_ACTUAL"
    ]
]

## 8.5 Mejor zona teórica por SKU

La mejor zona teórica minimiza el costo operativo, pero **todavía no considera capacidad ni límites de movimientos**.

In [ ]:
mejor_teorica = (
    matriz_sku_zona
    .sort_values(["SKU", "COSTO_NUEVO_MIN"])
    .groupby("SKU", as_index=False)
    .first()
)

mejor_teorica = mejor_teorica[
    [
        "SKU",
        "ZONA_ACTUAL",
        "ZONA",
        "N_LINEAS",
        "TIEMPO_LAYOUT_ACTUAL",
        "TIEMPO_MINUTOS",
        "CARGA_OPERATIVA_MIN",
        "COSTO_NUEVO_MIN",
        "AHORRO_MIN",
        "SCORE_PRIORIDAD",
        "RANKING_SCORE"
    ]
].rename(
    columns={
        "ZONA": "MEJOR_ZONA_TEORICA",
        "TIEMPO_MINUTOS": "MEJOR_TIEMPO_TEORICO",
        "AHORRO_MIN": "MEJOR_AHORRO_TEORICO"
    }
)

mejor_teorica.sort_values(
    "MEJOR_AHORRO_TEORICO",
    ascending=False
).head(20)

## 8.6 Validación de la matriz de escenarios

Cada SKU debe tener exactamente 9 alternativas y una de ellas debe corresponder a su zona actual.

In [ ]:
escenarios_por_sku = matriz_sku_zona.groupby("SKU").size()
zonas_actuales_en_matriz = matriz_sku_zona.groupby("SKU")["ES_ZONA_ACTUAL"].sum()

assert len(matriz_sku_zona) == len(base_maestra) * len(layout_limpio)
assert escenarios_por_sku.min() == len(layout_limpio)
assert escenarios_por_sku.max() == len(layout_limpio)
assert zonas_actuales_en_matriz.min() == 1
assert zonas_actuales_en_matriz.max() == 1

print("✅ Matriz SKU × Zona construida correctamente.")

# Fase 9 — Capacidad y ocupación

## 9.1 Volumen actual de los SKU de la muestra por zona

Calculamos cuánto volumen de los SKU modelados se encuentra actualmente en cada zona.

In [ ]:
volumen_actual_muestra = (
    base_maestra
    .groupby("ZONA_ACTUAL", as_index=False)["VOLUMEN_M3"]
    .sum()
    .rename(
        columns={
            "ZONA_ACTUAL": "ZONA",
            "VOLUMEN_M3": "VOLUMEN_MUESTRA_ACTUAL"
        }
    )
)

volumen_actual_muestra

## 9.2 Ocupación base no modelada

Para evitar contar dos veces el mismo volumen calculamos:

**Volumen base no modelado = Volumen usado reportado - Volumen de los SKU de la muestra**

Si la ocupación reportada corresponde únicamente a estos 100 SKU, este valor será aproximadamente cero.

In [ ]:
capacidad = ocupacion_limpio.merge(
    volumen_actual_muestra,
    on="ZONA",
    how="left"
)

capacidad["VOLUMEN_MUESTRA_ACTUAL"] = capacidad["VOLUMEN_MUESTRA_ACTUAL"].fillna(0)

capacidad["VOLUMEN_BASE_NO_MODELADO"] = (
    capacidad["VOLUMEN_USADO_M3"] -
    capacidad["VOLUMEN_MUESTRA_ACTUAL"]
).clip(lower=0)

capacidad[
    [
        "ZONA",
        "CAPACIDAD_MAX_M3",
        "VOLUMEN_USADO_M3",
        "VOLUMEN_MUESTRA_ACTUAL",
        "VOLUMEN_BASE_NO_MODELADO",
        "VOLUMEN_DISPONIBLE_M3"
    ]
]

## 9.3 Diagnóstico de capacidad

Esta celda muestra qué tan restrictiva es realmente la capacidad disponible.

Si las capacidades son muy superiores al volumen de la muestra, la restricción física a nivel de zona tendrá poco efecto. Esto señalaría una brecha de información: **capacidad real por ubicación o compatibilidad física**, no únicamente capacidad agregada de la zona.

In [ ]:
capacidad["USO_MODELO_ACTUAL_%"] = np.where(
    capacidad["CAPACIDAD_MAX_M3"] > 0,
    100 * capacidad["VOLUMEN_USADO_M3"] / capacidad["CAPACIDAD_MAX_M3"],
    0
)

capacidad[
    ["ZONA", "CAPACIDAD_MAX_M3", "VOLUMEN_USADO_M3", "USO_MODELO_ACTUAL_%"]
].sort_values("USO_MODELO_ACTUAL_%", ascending=False)

# Fase 10 — Configuración del optimizador

## 10.1 Parámetros de negocio

El modelo utilizará inicialmente:

- Máximo de movimientos: **20% de los SKU**.
- Penalización de movimiento: 0 minutos por defecto.
- Zonas no permitidas como destino: ninguna por defecto.

Estos parámetros son editables y deben ser validados con Operaciones.

In [ ]:
PORCENTAJE_MAX_MOVIMIENTO = 0.20

MAX_MOVIMIENTOS = max(
    1,
    int(round(len(base_maestra) * PORCENTAJE_MAX_MOVIMIENTO))
)

# Si Operaciones indica que una zona NO debe recibir nuevos SKU,
# agregar el nombre exacto aquí.
ZONAS_NO_DESTINO = []

# Penalización adicional por mover un SKU.
# Dejar 0 mientras no exista un costo real validado.
PENALIZACION_MOVIMIENTO = 0.0

print("Máximo de movimientos:", MAX_MOVIMIENTOS)
print("Zonas bloqueadas:", ZONAS_NO_DESTINO)
print("Penalización por movimiento:", PENALIZACION_MOVIMIENTO)

## 10.2 Preparación de diccionarios para optimización

Convertimos la información relevante a estructuras fáciles de utilizar en PuLP.

In [ ]:
lista_skus = base_maestra["SKU"].tolist()
lista_zonas = layout_limpio["ZONA"].tolist()

volumen_sku = base_maestra.set_index("SKU")["VOLUMEN_M3"].to_dict()
frecuencia_sku = base_maestra.set_index("SKU")["N_LINEAS"].to_dict()
zona_actual_sku = base_maestra.set_index("SKU")["ZONA_ACTUAL"].to_dict()

tiempo_zona = layout_limpio.set_index("ZONA")["TIEMPO_MINUTOS"].to_dict()

capacidad_max = capacidad.set_index("ZONA")["CAPACIDAD_MAX_M3"].to_dict()
ocupacion_no_modelada = capacidad.set_index("ZONA")["VOLUMEN_BASE_NO_MODELADO"].to_dict()

print("✅ Estructuras del optimizador preparadas.")

## 10.3 Creación del problema y variables binarias

Definimos:

**x(i,z) = 1** si el SKU `i` se asigna a la zona `z`.

**x(i,z) = 0** en caso contrario.

In [ ]:
modelo = pulp.LpProblem(
    "Optimizacion_Reslotting_CD_Aldeas",
    pulp.LpMinimize
)

x = pulp.LpVariable.dicts(
    "Asignacion",
    (lista_skus, lista_zonas),
    lowBound=0,
    upBound=1,
    cat="Binary"
)

print("✅ Problema de optimización creado.")

## 10.4 Función objetivo

Buscamos minimizar:

**Tiempo total de picking + penalización por movimientos**

El costo de picking de un SKU se aproxima mediante:

**N_LINEAS × Tiempo de la zona asignada**

In [ ]:
costo_picking = pulp.lpSum(
    frecuencia_sku[sku] *
    tiempo_zona[zona] *
    x[sku][zona]
    for sku in lista_skus
    for zona in lista_zonas
)

costo_movimientos = pulp.lpSum(
    PENALIZACION_MOVIMIENTO *
    x[sku][zona]
    for sku in lista_skus
    for zona in lista_zonas
    if zona != zona_actual_sku[sku]
)

modelo += costo_picking + costo_movimientos

print("✅ Función objetivo definida.")

## 10.5 Restricción: una zona por SKU

Cada SKU debe terminar asignado exactamente a una zona.

In [ ]:
for sku in lista_skus:
    modelo += (
        pulp.lpSum(x[sku][zona] for zona in lista_zonas) == 1
    )

print("✅ Restricción de asignación única agregada.")

## 10.6 Restricción de capacidad

La ocupación base más el volumen de los SKU asignados no puede superar la capacidad máxima de cada zona.

In [ ]:
for zona in lista_zonas:
    modelo += (
        ocupacion_no_modelada.get(zona, 0)
        +
        pulp.lpSum(
            volumen_sku[sku] * x[sku][zona]
            for sku in lista_skus
        )
        <= capacidad_max[zona]
    )

print("✅ Restricciones de capacidad agregadas.")

## 10.7 Restricción: máximo 20% de movimientos

Un SKU cuenta como movimiento cuando la zona asignada es diferente de su zona actual.

In [ ]:
modelo += (
    pulp.lpSum(
        x[sku][zona]
        for sku in lista_skus
        for zona in lista_zonas
        if zona != zona_actual_sku[sku]
    )
    <= MAX_MOVIMIENTOS
)

print("✅ Restricción de cantidad máxima de movimientos agregada.")

## 10.8 Zonas no permitidas como nuevo destino

Esta lógica permite bloquear destinos según reglas operativas futuras.

Un SKU que ya se encuentra en una zona bloqueada puede permanecer allí, pero otros SKU no podrán ingresar.

In [ ]:
for zona in ZONAS_NO_DESTINO:
    if zona not in lista_zonas:
        print(f"⚠️ Zona no encontrada en layout: {zona}")
        continue

    for sku in lista_skus:
        if zona_actual_sku[sku] != zona:
            modelo += x[sku][zona] == 0

print("✅ Restricciones de destinos bloqueados procesadas.")

# Fase 11 — Ejecución del optimizador

## 11.1 Resolver el modelo

Utilizamos el solver CBC incluido con PuLP.

Si el estado es `Optimal`, el modelo encontró la mejor solución posible dentro de las restricciones proporcionadas.

In [ ]:
solver = pulp.PULP_CBC_CMD(msg=False)
modelo.solve(solver)

estado = pulp.LpStatus[modelo.status]

print("Estado del modelo:", estado)
print("Valor de la función objetivo:", pulp.value(modelo.objective))

if estado != "Optimal":
    raise ValueError(
        "El optimizador no encontró una solución óptima. "
        "Revisar capacidades y restricciones."
    )

# Fase 12 — Recomendaciones por SKU

## 12.1 Extracción de la zona recomendada

Leemos qué variable binaria quedó activa para cada SKU.

In [ ]:
resultados = []

for sku in lista_skus:
    zona_nueva = None

    for zona in lista_zonas:
        valor = pulp.value(x[sku][zona])
        if valor is not None and valor > 0.5:
            zona_nueva = zona
            break

    resultados.append(
        {
            "SKU": sku,
            "ZONA_RECOMENDADA": zona_nueva
        }
    )

recomendaciones = pd.DataFrame(resultados)

recomendaciones = base_maestra.merge(
    recomendaciones,
    on="SKU",
    how="left"
)

print("✅ Recomendaciones extraídas.")

## 12.2 Cálculo del impacto antes y después

Calculamos:

- Tiempo/costo actual.
- Tiempo/costo optimizado.
- Ahorro estimado.
- Porcentaje de ahorro.
- Acción: MOVER o MANTENER.

In [ ]:
tiempos_dict = layout_limpio.set_index("ZONA")["TIEMPO_MINUTOS"].to_dict()

recomendaciones["TIEMPO_NUEVO_MIN"] = (
    recomendaciones["ZONA_RECOMENDADA"].map(tiempos_dict)
)

recomendaciones["COSTO_ACTUAL_MIN"] = (
    recomendaciones["N_LINEAS"] *
    recomendaciones["TIEMPO_LAYOUT_ACTUAL"]
)

recomendaciones["COSTO_NUEVO_MIN"] = (
    recomendaciones["N_LINEAS"] *
    recomendaciones["TIEMPO_NUEVO_MIN"]
)

recomendaciones["AHORRO_ESTIMADO_MIN"] = (
    recomendaciones["COSTO_ACTUAL_MIN"] -
    recomendaciones["COSTO_NUEVO_MIN"]
)

recomendaciones["AHORRO_%"] = np.where(
    recomendaciones["COSTO_ACTUAL_MIN"] > 0,
    100 *
    recomendaciones["AHORRO_ESTIMADO_MIN"] /
    recomendaciones["COSTO_ACTUAL_MIN"],
    0
)

recomendaciones["MOVIMIENTO"] = np.where(
    recomendaciones["ZONA_ACTUAL"] != recomendaciones["ZONA_RECOMENDADA"],
    "MOVER",
    "MANTENER"
)

recomendaciones[
    [
        "SKU",
        "ZONA_ACTUAL",
        "ZONA_RECOMENDADA",
        "N_LINEAS",
        "COSTO_ACTUAL_MIN",
        "COSTO_NUEVO_MIN",
        "AHORRO_ESTIMADO_MIN",
        "AHORRO_%",
        "MOVIMIENTO"
    ]
].sort_values("AHORRO_ESTIMADO_MIN", ascending=False).head(20)

## 12.3 Justificación automática

Generamos una explicación legible para cada recomendación.

In [ ]:
def generar_justificacion(fila):
    if fila["MOVIMIENTO"] == "MOVER":
        return (
            f"Mover de {fila['ZONA_ACTUAL']} a {fila['ZONA_RECOMENDADA']}. "
            f"El SKU registró {int(fila['N_LINEAS'])} visitas. "
            f"El tiempo de acceso de zona pasa de "
            f"{fila['TIEMPO_LAYOUT_ACTUAL']:.2f} a "
            f"{fila['TIEMPO_NUEVO_MIN']:.2f} min. "
            f"Ahorro estimado en la muestra: "
            f"{fila['AHORRO_ESTIMADO_MIN']:.2f} min."
        )

    return (
        f"Mantener en {fila['ZONA_ACTUAL']}. "
        f"Dentro de las restricciones actuales, el optimizador "
        f"no seleccionó un cambio de zona."
    )

recomendaciones["JUSTIFICACION"] = recomendaciones.apply(
    generar_justificacion,
    axis=1
)

print("✅ Justificaciones generadas.")

## 12.4 Tabla final ordenada

Ordenamos los resultados por ahorro estimado.

In [ ]:
columnas_resultado = [
    "RANKING_SCORE",
    "SKU",
    "MARCA",
    "FAMILIA",
    "ABC",
    "ROTACION_6M",
    "N_PEDIDOS",
    "N_LINEAS",
    "CANT_TOTAL",
    "VOLUMEN_M3",
    "PESO_KG",
    "ZONA_ACTUAL",
    "ZONA_RECOMENDADA",
    "TIEMPO_LAYOUT_ACTUAL",
    "TIEMPO_NUEVO_MIN",
    "COSTO_ACTUAL_MIN",
    "COSTO_NUEVO_MIN",
    "AHORRO_ESTIMADO_MIN",
    "AHORRO_%",
    "SCORE_PRIORIDAD",
    "MOVIMIENTO",
    "JUSTIFICACION"
]

resultado_final = (
    recomendaciones[columnas_resultado]
    .sort_values("AHORRO_ESTIMADO_MIN", ascending=False)
    .reset_index(drop=True)
)

movimientos_recomendados = (
    resultado_final[
        resultado_final["MOVIMIENTO"] == "MOVER"
    ]
    .reset_index(drop=True)
)

movimientos_recomendados

# Fase 13 — KPI del MVP

## 13.1 Indicadores globales

Calculamos el impacto del escenario optimizado frente a la situación actual.

In [ ]:
tiempo_total_actual = recomendaciones["COSTO_ACTUAL_MIN"].sum()
tiempo_total_nuevo = recomendaciones["COSTO_NUEVO_MIN"].sum()

ahorro_total = tiempo_total_actual - tiempo_total_nuevo

ahorro_porcentaje = (
    100 * ahorro_total / tiempo_total_actual
    if tiempo_total_actual > 0
    else 0
)

cantidad_movimientos = recomendaciones["MOVIMIENTO"].eq("MOVER").sum()
porcentaje_movimientos = 100 * cantidad_movimientos / len(recomendaciones)

print("=" * 55)
print("RESULTADOS DEL MVP")
print("=" * 55)
print(f"SKU analizados: {len(recomendaciones)}")
print(f"SKU movidos: {cantidad_movimientos}")
print(f"% SKU movidos: {porcentaje_movimientos:.2f}%")
print(f"Tiempo actual estimado: {tiempo_total_actual:.2f} min")
print(f"Tiempo optimizado estimado: {tiempo_total_nuevo:.2f} min")
print(f"Ahorro estimado: {ahorro_total:.2f} min")
print(f"Reducción estimada: {ahorro_porcentaje:.2f}%")

## 13.2 Resumen de KPI en tabla

In [ ]:
resumen_kpi = pd.DataFrame(
    {
        "INDICADOR": [
            "SKU analizados",
            "SKU movidos",
            "% SKU movidos",
            "Tiempo actual estimado (min)",
            "Tiempo optimizado estimado (min)",
            "Ahorro estimado (min)",
            "Reducción estimada (%)"
        ],
        "VALOR": [
            len(recomendaciones),
            cantidad_movimientos,
            porcentaje_movimientos,
            tiempo_total_actual,
            tiempo_total_nuevo,
            ahorro_total,
            ahorro_porcentaje
        ]
    }
)

resumen_kpi

# Fase 14 — Validación de factibilidad

## 14.1 Capacidad final por zona

Recalculamos el volumen final de cada zona después del reslotting para verificar que no se exceda la capacidad máxima.

In [ ]:
volumen_nuevo = (
    recomendaciones
    .groupby("ZONA_RECOMENDADA", as_index=False)["VOLUMEN_M3"]
    .sum()
    .rename(
        columns={
            "ZONA_RECOMENDADA": "ZONA",
            "VOLUMEN_M3": "VOLUMEN_SKU_ASIGNADOS"
        }
    )
)

validacion_capacidad = capacidad.merge(
    volumen_nuevo,
    on="ZONA",
    how="left"
)

validacion_capacidad["VOLUMEN_SKU_ASIGNADOS"] = (
    validacion_capacidad["VOLUMEN_SKU_ASIGNADOS"].fillna(0)
)

validacion_capacidad["VOLUMEN_FINAL_M3"] = (
    validacion_capacidad["VOLUMEN_BASE_NO_MODELADO"] +
    validacion_capacidad["VOLUMEN_SKU_ASIGNADOS"]
)

validacion_capacidad["CAPACIDAD_OK"] = (
    validacion_capacidad["VOLUMEN_FINAL_M3"] <=
    validacion_capacidad["CAPACIDAD_MAX_M3"] + 1e-9
)

validacion_capacidad[
    [
        "ZONA",
        "CAPACIDAD_MAX_M3",
        "VOLUMEN_FINAL_M3",
        "CAPACIDAD_OK"
    ]
]

## 14.2 Validaciones automáticas

Comprobamos:

- Todas las zonas dentro de capacidad.
- Máximo de movimientos respetado.
- Todos los SKU con zona recomendada.

In [ ]:
assert validacion_capacidad["CAPACIDAD_OK"].all(), "Hay una zona que supera capacidad."
assert cantidad_movimientos <= MAX_MOVIMIENTOS, "Se superó el máximo de movimientos."
assert recomendaciones["ZONA_RECOMENDADA"].notna().all(), "Hay SKU sin zona recomendada."

print("✅ Todas las restricciones principales se cumplen.")

## 14.3 Diagnóstico de concentración de destinos

Si muchos movimientos terminan en una sola zona, esto puede indicar que las capacidades agregadas actuales son demasiado amplias para representar la realidad del almacén.

En ese caso será necesario incorporar información más granular, por ejemplo:

- Capacidad real por ubicación.
- Compatibilidad física SKU-zona.
- Equipos disponibles por zona.
- Restricciones ergonómicas.
- Reglas FIFO específicas.

In [ ]:
destinos_movimientos = (
    movimientos_recomendados["ZONA_RECOMENDADA"]
    .value_counts()
    .rename_axis("ZONA_RECOMENDADA")
    .reset_index(name="SKU_MOVIDOS")
)

destinos_movimientos

# Fase 15 — Visualización

## 15.1 Top movimientos por ahorro estimado

In [ ]:
top_grafico = (
    movimientos_recomendados
    .head(20)
    .sort_values("AHORRO_ESTIMADO_MIN", ascending=True)
)

plt.figure(figsize=(10, 7))
plt.barh(
    top_grafico["SKU"],
    top_grafico["AHORRO_ESTIMADO_MIN"]
)
plt.title("Top SKU por ahorro estimado de tiempo")
plt.xlabel("Ahorro estimado (minutos)")
plt.ylabel("SKU")
plt.tight_layout()
plt.show()

## 15.2 Comparación Actual vs. Optimizado

In [ ]:
comparacion = pd.DataFrame(
    {
        "ESCENARIO": ["Actual", "Optimizado"],
        "TIEMPO_MIN": [tiempo_total_actual, tiempo_total_nuevo]
    }
)

plt.figure(figsize=(6, 5))
plt.bar(
    comparacion["ESCENARIO"],
    comparacion["TIEMPO_MIN"]
)
plt.ylabel("Tiempo estimado (minutos)")
plt.title("Tiempo de picking: Actual vs. Optimizado")
plt.tight_layout()
plt.show()

comparacion

## 15.3 Distribución de movimientos por zona destino

In [ ]:
if len(destinos_movimientos) > 0:
    destinos_plot = destinos_movimientos.sort_values("SKU_MOVIDOS", ascending=True)

    plt.figure(figsize=(9, 5))
    plt.barh(
        destinos_plot["ZONA_RECOMENDADA"],
        destinos_plot["SKU_MOVIDOS"]
    )
    plt.xlabel("Cantidad de SKU movidos")
    plt.ylabel("Zona destino")
    plt.title("Distribución de movimientos por zona recomendada")
    plt.tight_layout()
    plt.show()
else:
    print("No existen movimientos recomendados.")

# Fase 16 — Exportación de resultados

## 16.1 Generación del Excel final

El archivo contendrá:

- `BASE_SKU`
- `RANKING`
- `MATRIZ_SKU_ZONA`
- `MEJOR_TEORICA`
- `RECOMENDACIONES`
- `MOVIMIENTOS`
- `CAPACIDAD_FINAL`
- `RESUMEN`

In [ ]:
ARCHIVO_SALIDA = "resultado_reslotting_inchcape.xlsx"

with pd.ExcelWriter(
    ARCHIVO_SALIDA,
    engine="openpyxl"
) as writer:

    base_maestra.to_excel(
        writer,
        sheet_name="BASE_SKU",
        index=False
    )

    ranking_score.to_excel(
        writer,
        sheet_name="RANKING",
        index=False
    )

    matriz_sku_zona.to_excel(
        writer,
        sheet_name="MATRIZ_SKU_ZONA",
        index=False
    )

    mejor_teorica.to_excel(
        writer,
        sheet_name="MEJOR_TEORICA",
        index=False
    )

    resultado_final.to_excel(
        writer,
        sheet_name="RECOMENDACIONES",
        index=False
    )

    movimientos_recomendados.to_excel(
        writer,
        sheet_name="MOVIMIENTOS",
        index=False
    )

    validacion_capacidad.to_excel(
        writer,
        sheet_name="CAPACIDAD_FINAL",
        index=False
    )

    resumen_kpi.to_excel(
        writer,
        sheet_name="RESUMEN",
        index=False
    )

print("✅ Archivo generado:", ARCHIVO_SALIDA)

## 16.2 Descargar el Excel generado

In [ ]:
from google.colab import files

files.download(ARCHIVO_SALIDA)

# Fase 17 — Salida ejecutiva

## 17.1 Tabla para dashboard o exposición

Esta tabla resume únicamente los movimientos seleccionados por el optimizador.

In [ ]:
salida_ejecutiva = movimientos_recomendados[
    [
        "RANKING_SCORE",
        "SKU",
        "MARCA",
        "FAMILIA",
        "ABC",
        "ROTACION_6M",
        "N_LINEAS",
        "ZONA_ACTUAL",
        "ZONA_RECOMENDADA",
        "TIEMPO_LAYOUT_ACTUAL",
        "TIEMPO_NUEVO_MIN",
        "AHORRO_ESTIMADO_MIN",
        "AHORRO_%",
        "SCORE_PRIORIDAD",
        "JUSTIFICACION"
    ]
].copy()

salida_ejecutiva = salida_ejecutiva.sort_values(
    "AHORRO_ESTIMADO_MIN",
    ascending=False
).reset_index(drop=True)

salida_ejecutiva

## 17.2 Conclusiones automáticas del escenario

Esta celda resume los principales resultados para facilitar su interpretación.

In [ ]:
print("=" * 65)
print("CONCLUSIONES DEL ESCENARIO")
print("=" * 65)

print(
    f"1. Se analizaron {len(recomendaciones)} SKU y "
    f"el optimizador propone mover {cantidad_movimientos}."
)

print(
    f"2. El tiempo estimado de picking pasa de "
    f"{tiempo_total_actual:.2f} a {tiempo_total_nuevo:.2f} minutos."
)

print(
    f"3. El ahorro estimado en la muestra es de "
    f"{ahorro_total:.2f} minutos ({ahorro_porcentaje:.2f}%)."
)

if len(destinos_movimientos) > 0:
    destino_principal = destinos_movimientos.iloc[0]["ZONA_RECOMENDADA"]
    cantidad_destino = int(destinos_movimientos.iloc[0]["SKU_MOVIDOS"])

    print(
        f"4. La zona que recibe más movimientos es "
        f"{destino_principal}, con {cantidad_destino} SKU."
    )

    if cantidad_movimientos > 0 and cantidad_destino / cantidad_movimientos >= 0.60:
        print(
            "5. Existe una alta concentración de movimientos en una sola zona. "
            "Esto sugiere validar capacidad real por ubicación y compatibilidad "
            "física antes de implementar la recomendación."
        )

print(
    "\nNota: la solución representa la mejor decisión factible con las "
    "variables y restricciones disponibles en el dataset actual."
)

# Siguiente evolución — ML predictivo

El notebook actual resuelve el **MVP de reslotting con scoring y optimización**.

Para incorporar Machine Learning predictivo se necesita una fuente histórica con fechas, idealmente de 12 a 24 meses, por ejemplo:

| FECHA | SKU | PEDIDOS | UNIDADES |
|---|---|---:|---:|
| 2025-01 | SKU00001 | ... | ... |
| 2025-02 | SKU00001 | ... | ... |

Con esa información se podría entrenar un modelo para predecir:

- Próxima demanda.
- Próxima frecuencia de picks.
- Cambios de patrón por SKU.

La predicción sustituiría la frecuencia observada actual dentro de la función de costo del optimizador, evolucionando hacia un **reslotting dinámico y predictivo**.

## Restricciones todavía pendientes de información

El dataset actual no permite modelar completamente:

- FIFO por lote/fecha.
- Compatibilidad física SKU-zona.
- Equipos específicos por zona.
- Restricciones ergonómicas.
- Capacidad real por ubicación individual.
- Costo real de realizar un movimiento.

Estas variables deben incorporarse cuando Operaciones las valide o provea.

---
# Extensión Machine Learning — Fases 18 a 27

A partir de este punto añadimos un componente real de **Machine Learning no supervisado**.

## ¿Por qué K-Means?

El dataset actual no contiene:

- Una columna objetivo con la **zona óptima real** de cada SKU.
- Fechas suficientes para construir una serie temporal de demanda.
- Etiquetas históricas de decisiones correctas/incorrectas de reslotting.

Por lo tanto, no sería metodológicamente correcto entrenar un modelo supervisado para “predecir la zona óptima”.

En cambio, **K-Means** permite descubrir automáticamente grupos de SKU con comportamientos operativos similares. Estos perfiles complementarán el Score y las recomendaciones del optimizador.

La arquitectura final será:

**Datos → Ingeniería de variables → K-Means → Perfil ML → Score → Optimización → Recomendación**

# Fase 18 — Preparación del dataset de Machine Learning

## 18.1 Importación de herramientas de ML

Utilizaremos herramientas de `scikit-learn` para:

- Escalar variables.
- Entrenar K-Means.
- Evaluar la calidad de los clusters.
- Reducir dimensionalidad mediante PCA únicamente para visualización.
- Guardar posteriormente el modelo entrenado.

El modelo K-Means será reproducible utilizando `random_state=42`.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)
from sklearn.decomposition import PCA
import joblib

print("✅ Herramientas de Machine Learning importadas correctamente.")

## 18.2 Selección de variables para K-Means

El modelo utilizará variables que describen el comportamiento físico y operativo de cada SKU:

- `ROTACION_6M`
- `N_LINEAS`
- `N_PEDIDOS`
- `CANT_TOTAL`
- `VOLUMEN_M3`
- `PESO_KG`
- `TIEMPO_LAYOUT_ACTUAL`
- `CARGA_OPERATIVA_MIN`
- `AHORRO_TEORICO_MIN`

No se incluye `SKU`, marca, familia, ABC ni zona como números artificiales.

`ABC` tampoco se transforma en una etiqueta numérica para K-Means, porque ya tenemos `ROTACION_6M` como variable cuantitativa y queremos evitar imponer una distancia artificial entre categorías.

In [ ]:
variables_ml = [
    "ROTACION_6M",
    "N_LINEAS",
    "N_PEDIDOS",
    "CANT_TOTAL",
    "VOLUMEN_M3",
    "PESO_KG",
    "TIEMPO_LAYOUT_ACTUAL",
    "CARGA_OPERATIVA_MIN",
    "AHORRO_TEORICO_MIN"
]

X_ml_original = base_maestra[variables_ml].copy()

print("Variables utilizadas por el modelo:")
for variable in variables_ml:
    print("-", variable)

print("\nDimensión del dataset ML:", X_ml_original.shape)
X_ml_original.head()

## 18.3 Validación de calidad antes del entrenamiento

K-Means no admite valores faltantes.

Antes de entrenar verificamos:

- Valores nulos.
- Valores infinitos.
- Variables sin variación.

Una variable sin variación no aporta información al clustering y debe excluirse.

In [ ]:
print("Valores faltantes por variable:")
print(X_ml_original.isna().sum())

print("\nValores infinitos:")
print(np.isinf(X_ml_original.select_dtypes(include=np.number)).sum())

variables_sin_variacion = [
    col for col in variables_ml
    if X_ml_original[col].nunique(dropna=False) <= 1
]

print("\nVariables sin variación:", variables_sin_variacion)

if variables_sin_variacion:
    variables_ml = [
        col for col in variables_ml
        if col not in variables_sin_variacion
    ]
    X_ml_original = base_maestra[variables_ml].copy()
    print("Se eliminaron automáticamente las variables sin variación.")

assert X_ml_original.isna().sum().sum() == 0, (
    "Existen valores faltantes. Deben corregirse antes de entrenar."
)

assert np.isfinite(X_ml_original.to_numpy(dtype=float)).all(), (
    "Existen valores infinitos en las variables ML."
)

print("\n✅ Dataset listo para Machine Learning.")

# Fase 19 — Escalado de variables

## 19.1 StandardScaler

K-Means utiliza distancias entre observaciones.

Por este motivo, una variable con valores numéricamente grandes podría dominar artificialmente al resto.

Utilizaremos `StandardScaler`, que transforma cada variable para que tenga aproximadamente:

- Media = 0
- Desviación estándar = 1

El escalado se ajusta sobre los 100 SKU porque este es un problema de aprendizaje no supervisado sobre la muestra completa.

In [ ]:
scaler = StandardScaler()

X_ml_scaled = scaler.fit_transform(X_ml_original)

X_ml_scaled_df = pd.DataFrame(
    X_ml_scaled,
    columns=variables_ml,
    index=base_maestra.index
)

print("✅ Variables escaladas correctamente.")
X_ml_scaled_df.head()

## 19.2 Comprobación del escalado

Verificamos que las medias estén cercanas a 0 y las desviaciones estándar cercanas a 1.

In [ ]:
resumen_escalado = pd.DataFrame({
    "MEDIA": X_ml_scaled_df.mean(),
    "DESV_STD": X_ml_scaled_df.std(ddof=0)
})

resumen_escalado

# Fase 20 — Selección del número de clusters

## 20.1 Evaluación de diferentes valores de K

K-Means necesita que definamos cuántos clusters deseamos encontrar.

No elegiremos este número arbitrariamente.

Probaremos valores entre **K = 2 y K = 8** y evaluaremos:

- **Inercia:** menor es mejor, aunque siempre disminuye al aumentar K.
- **Silhouette Score:** valores más altos indican mejor separación y cohesión.

Seleccionaremos automáticamente el K con mayor Silhouette Score dentro del rango probado.

In [ ]:
resultados_k = []

max_k = min(8, len(base_maestra) - 1)

for k in range(2, max_k + 1):
    modelo_k = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    etiquetas_k = modelo_k.fit_predict(X_ml_scaled)

    if len(np.unique(etiquetas_k)) > 1:
        silueta = silhouette_score(X_ml_scaled, etiquetas_k)
    else:
        silueta = np.nan

    resultados_k.append({
        "K": k,
        "INERCIA": modelo_k.inertia_,
        "SILHOUETTE": silueta
    })

evaluacion_k = pd.DataFrame(resultados_k)

evaluacion_k

## 20.2 Gráfico del método del codo

La inercia permite observar en qué punto agregar más clusters comienza a producir mejoras cada vez menores.

Este gráfico sirve como apoyo visual; la selección automática se basará principalmente en Silhouette Score.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(
    evaluacion_k["K"],
    evaluacion_k["INERCIA"],
    marker="o"
)
plt.xlabel("Número de clusters (K)")
plt.ylabel("Inercia")
plt.title("Método del codo — K-Means")
plt.xticks(evaluacion_k["K"])
plt.tight_layout()
plt.show()

## 20.3 Gráfico de Silhouette Score

El Silhouette Score se encuentra aproximadamente entre -1 y 1.

Interpretación general:

- Cercano a 1: clusters bien separados.
- Cercano a 0: clusters parcialmente superpuestos.
- Negativo: posibles asignaciones poco adecuadas.

No existe un umbral universal; el resultado debe interpretarse junto con el contexto operativo.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(
    evaluacion_k["K"],
    evaluacion_k["SILHOUETTE"],
    marker="o"
)
plt.xlabel("Número de clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Selección de K mediante Silhouette Score")
plt.xticks(evaluacion_k["K"])
plt.tight_layout()
plt.show()

## 20.4 Selección automática del mejor K

Seleccionamos el valor de K con mayor Silhouette Score.

Este criterio evita fijar manualmente tres o cuatro perfiles sin evidencia en los datos.

In [ ]:
mejor_fila_k = evaluacion_k.loc[
    evaluacion_k["SILHOUETTE"].idxmax()
]

MEJOR_K = int(mejor_fila_k["K"])

print("✅ Mejor K según Silhouette Score:", MEJOR_K)
print(
    "Silhouette:",
    round(float(mejor_fila_k["SILHOUETTE"]), 4)
)

# Fase 21 — Entrenamiento del modelo K-Means

## 21.1 Entrenamiento final

Entrenamos K-Means utilizando el número de clusters seleccionado en la fase anterior.

El resultado será una etiqueta `CLUSTER_ML` para cada SKU.

La numeración del cluster (0, 1, 2, ...) no representa por sí misma prioridad. La prioridad será interpretada posteriormente utilizando el perfil operativo de cada grupo.

In [ ]:
modelo_kmeans = KMeans(
    n_clusters=MEJOR_K,
    random_state=42,
    n_init=20
)

base_maestra["CLUSTER_ML"] = modelo_kmeans.fit_predict(
    X_ml_scaled
)

print("✅ K-Means entrenado.")
print("\nCantidad de SKU por cluster:")
print(
    base_maestra["CLUSTER_ML"]
    .value_counts()
    .sort_index()
)

## 21.2 Distancia al centroide

Calculamos para cada SKU su distancia al centro del cluster asignado.

Una distancia menor significa que el SKU representa de forma más típica el perfil de su cluster.

Una distancia elevada puede indicar un SKU menos representativo o más extremo dentro del grupo.

In [ ]:
distancias_centroides = modelo_kmeans.transform(X_ml_scaled)

base_maestra["DISTANCIA_CENTROIDE"] = [
    distancias_centroides[i, cluster]
    for i, cluster in enumerate(base_maestra["CLUSTER_ML"])
]

base_maestra[
    ["SKU", "CLUSTER_ML", "DISTANCIA_CENTROIDE"]
].sort_values("DISTANCIA_CENTROIDE", ascending=False).head(15)

# Fase 22 — Evaluación del modelo ML

## 22.1 Métricas de clustering

Como K-Means es un modelo no supervisado, no utilizamos accuracy, precision o recall.

Evaluaremos:

### Silhouette Score
Mide cohesión interna y separación entre clusters. Mayor es mejor.

### Calinski-Harabasz
Compara dispersión entre clusters frente a dispersión interna. Mayor es mejor.

### Davies-Bouldin
Evalúa similitud entre clusters. Menor es mejor.

Estas métricas permiten evaluar la estructura encontrada, pero no reemplazan la validación operativa.

In [ ]:
etiquetas_finales = base_maestra["CLUSTER_ML"].to_numpy()

silhouette_final = silhouette_score(
    X_ml_scaled,
    etiquetas_finales
)

calinski_final = calinski_harabasz_score(
    X_ml_scaled,
    etiquetas_finales
)

davies_final = davies_bouldin_score(
    X_ml_scaled,
    etiquetas_finales
)

metricas_ml = pd.DataFrame({
    "METRICA": [
        "Número de clusters",
        "Silhouette Score",
        "Calinski-Harabasz",
        "Davies-Bouldin",
        "Inercia"
    ],
    "VALOR": [
        MEJOR_K,
        silhouette_final,
        calinski_final,
        davies_final,
        modelo_kmeans.inertia_
    ]
})

metricas_ml

## 22.2 Interpretación automática básica de Silhouette

Esta interpretación es orientativa y no debe considerarse una regla universal.

El criterio final debe combinar la métrica con la utilidad operacional de los perfiles encontrados.

In [ ]:
if silhouette_final >= 0.50:
    interpretacion_silhouette = "Separación de clusters relativamente clara."
elif silhouette_final >= 0.25:
    interpretacion_silhouette = (
        "Estructura de clusters moderada; requiere validación operativa."
    )
else:
    interpretacion_silhouette = (
        "Separación débil; los perfiles deben interpretarse con cautela."
    )

print("Silhouette Score:", round(silhouette_final, 4))
print("Interpretación:", interpretacion_silhouette)

# Fase 23 — Interpretación de los clusters

## 23.1 Perfil promedio por cluster

K-Means entrega grupos numéricos, pero necesitamos traducirlos a información útil.

Calcularemos los valores promedio de las variables originales dentro de cada cluster.

In [ ]:
perfil_clusters = (
    base_maestra
    .groupby("CLUSTER_ML")[variables_ml]
    .mean()
)

perfil_clusters["CANTIDAD_SKU"] = (
    base_maestra
    .groupby("CLUSTER_ML")
    .size()
)

perfil_clusters.reset_index()

## 23.2 Construcción de un índice de impacto del cluster

La etiqueta 0, 1, 2, etc. no tiene significado de prioridad.

Para interpretar los grupos calcularemos un índice descriptivo basado en cuatro variables directamente relacionadas con impacto:

- Carga operativa.
- Ahorro teórico.
- Frecuencia de visitas.
- Rotación.

Este índice **no vuelve a entrenar K-Means**; solo sirve para ordenar e interpretar los clusters ya descubiertos.

In [ ]:
variables_impacto_cluster = [
    "CARGA_OPERATIVA_MIN",
    "AHORRO_TEORICO_MIN",
    "N_LINEAS",
    "ROTACION_6M"
]

perfil_impacto = (
    base_maestra
    .groupby("CLUSTER_ML")[variables_impacto_cluster]
    .mean()
    .reset_index()
)

for col in variables_impacto_cluster:
    perfil_impacto[col + "_NORM"] = normalizar_01(
        perfil_impacto[col]
    )

perfil_impacto["INDICE_IMPACTO_CLUSTER"] = (
    perfil_impacto[
        [c + "_NORM" for c in variables_impacto_cluster]
    ]
    .mean(axis=1)
)

perfil_impacto["PRIORIDAD_CLUSTER_RANK"] = (
    perfil_impacto["INDICE_IMPACTO_CLUSTER"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

perfil_impacto.sort_values(
    "PRIORIDAD_CLUSTER_RANK"
)

## 23.3 Etiquetas interpretables de perfil

Para facilitar la presentación clasificamos los clusters en tres niveles descriptivos:

- **Impacto alto**
- **Impacto medio**
- **Impacto bajo**

La etiqueta se deriva del ranking relativo entre clusters y no es una verdad absoluta del negocio.

In [ ]:
def etiquetar_perfil(rank, total_clusters):
    proporcion = rank / total_clusters

    if proporcion <= 1/3:
        return "Impacto alto"
    elif proporcion <= 2/3:
        return "Impacto medio"
    else:
        return "Impacto bajo"

perfil_impacto["PERFIL_ML"] = perfil_impacto[
    "PRIORIDAD_CLUSTER_RANK"
].apply(
    lambda r: etiquetar_perfil(r, MEJOR_K)
)

perfil_impacto[
    [
        "CLUSTER_ML",
        "PRIORIDAD_CLUSTER_RANK",
        "INDICE_IMPACTO_CLUSTER",
        "PERFIL_ML"
    ]
].sort_values("PRIORIDAD_CLUSTER_RANK")

## 23.4 Tabla completa de perfiles

Unimos el resumen descriptivo de K-Means con la etiqueta de impacto.

In [ ]:
perfil_clusters_final = (
    perfil_clusters
    .reset_index()
    .merge(
        perfil_impacto[
            [
                "CLUSTER_ML",
                "INDICE_IMPACTO_CLUSTER",
                "PRIORIDAD_CLUSTER_RANK",
                "PERFIL_ML"
            ]
        ],
        on="CLUSTER_ML",
        how="left"
    )
)

perfil_clusters_final.sort_values(
    "PRIORIDAD_CLUSTER_RANK"
)

# Fase 24 — Integración del ML con el motor de decisión

## 24.1 Incorporación del perfil ML a cada SKU

Asignamos a cada SKU:

- Cluster.
- Perfil ML.
- Ranking de impacto del cluster.
- Distancia al centroide.

De esta manera, el modelo ML funciona como una capa de segmentación y explicabilidad.

In [ ]:
mapa_perfil = perfil_impacto.set_index(
    "CLUSTER_ML"
)["PERFIL_ML"].to_dict()

mapa_rank_cluster = perfil_impacto.set_index(
    "CLUSTER_ML"
)["PRIORIDAD_CLUSTER_RANK"].to_dict()

mapa_indice_cluster = perfil_impacto.set_index(
    "CLUSTER_ML"
)["INDICE_IMPACTO_CLUSTER"].to_dict()

base_maestra["PERFIL_ML"] = base_maestra[
    "CLUSTER_ML"
].map(mapa_perfil)

base_maestra["PRIORIDAD_CLUSTER_RANK"] = base_maestra[
    "CLUSTER_ML"
].map(mapa_rank_cluster)

base_maestra["INDICE_IMPACTO_CLUSTER"] = base_maestra[
    "CLUSTER_ML"
].map(mapa_indice_cluster)

base_maestra[
    [
        "SKU",
        "CLUSTER_ML",
        "PERFIL_ML",
        "PRIORIDAD_CLUSTER_RANK",
        "INDICE_IMPACTO_CLUSTER",
        "SCORE_PRIORIDAD",
        "RANKING_SCORE"
    ]
].head(20)

## 24.2 Actualización de recomendaciones con información ML

El optimizador ya calculó la zona recomendada.

Ahora añadimos el perfil aprendido por K-Means a esa recomendación, sin alterar las restricciones matemáticas.

Esto mantiene una separación metodológica clara:

- **K-Means:** descubre perfiles.
- **Score:** prioriza.
- **PuLP:** decide una asignación factible.

In [ ]:
columnas_ml_para_unir = [
    "SKU",
    "CLUSTER_ML",
    "PERFIL_ML",
    "PRIORIDAD_CLUSTER_RANK",
    "INDICE_IMPACTO_CLUSTER",
    "DISTANCIA_CENTROIDE"
]

info_ml = base_maestra[
    columnas_ml_para_unir
].drop_duplicates("SKU")

# Evitar columnas duplicadas si se ejecuta más de una vez.
for col in columnas_ml_para_unir[1:]:
    if col in recomendaciones.columns:
        recomendaciones = recomendaciones.drop(columns=col)

recomendaciones = recomendaciones.merge(
    info_ml,
    on="SKU",
    how="left"
)

print("✅ Información ML integrada a las recomendaciones.")

# Fase 25 — Resultado híbrido ML + Score + Optimización

## 25.1 Tabla híbrida final

Construimos una salida que muestre conjuntamente:

- Perfil ML.
- Score.
- Ranking.
- Zona actual.
- Zona recomendada.
- Ahorro.
- Acción.

El ML no reemplaza la optimización: ambos componentes cumplen funciones diferentes dentro del sistema.

In [ ]:
columnas_hibridas = [
    "SKU",
    "MARCA",
    "FAMILIA",
    "ABC",
    "ROTACION_6M",
    "N_LINEAS",
    "N_PEDIDOS",
    "CANT_TOTAL",
    "CLUSTER_ML",
    "PERFIL_ML",
    "PRIORIDAD_CLUSTER_RANK",
    "DISTANCIA_CENTROIDE",
    "SCORE_PRIORIDAD",
    "RANKING_SCORE",
    "ZONA_ACTUAL",
    "ZONA_RECOMENDADA",
    "TIEMPO_LAYOUT_ACTUAL",
    "TIEMPO_NUEVO_MIN",
    "AHORRO_ESTIMADO_MIN",
    "AHORRO_%",
    "MOVIMIENTO",
    "JUSTIFICACION"
]

resultado_hibrido = (
    recomendaciones[columnas_hibridas]
    .sort_values(
        ["MOVIMIENTO", "AHORRO_ESTIMADO_MIN"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

resultado_hibrido.head(25)

## 25.2 Comportamiento del optimizador por cluster

Analizamos cuántos SKU de cada cluster fueron seleccionados para movimiento y cuánto ahorro generan en promedio.

In [ ]:
analisis_cluster_optimizacion = (
    recomendaciones
    .groupby(["CLUSTER_ML", "PERFIL_ML"])
    .agg(
        SKU=("SKU", "count"),
        SCORE_PROMEDIO=("SCORE_PRIORIDAD", "mean"),
        AHORRO_PROMEDIO=("AHORRO_ESTIMADO_MIN", "mean"),
        AHORRO_TOTAL=("AHORRO_ESTIMADO_MIN", "sum"),
        MOVIMIENTOS=("MOVIMIENTO", lambda s: (s == "MOVER").sum())
    )
    .reset_index()
)

analisis_cluster_optimizacion["TASA_MOVIMIENTO_%"] = (
    100 *
    analisis_cluster_optimizacion["MOVIMIENTOS"] /
    analisis_cluster_optimizacion["SKU"]
)

analisis_cluster_optimizacion.sort_values(
    "AHORRO_TOTAL",
    ascending=False
)

## 25.3 Justificación híbrida

Añadimos una explicación que combine el perfil ML con la recomendación del optimizador.

In [ ]:
def generar_justificacion_hibrida(fila):
    perfil = fila["PERFIL_ML"]

    if fila["MOVIMIENTO"] == "MOVER":
        return (
            f"El SKU pertenece al perfil ML '{perfil}' y obtuvo "
            f"un Score de Prioridad de {fila['SCORE_PRIORIDAD']:.2f}. "
            f"El optimizador recomienda moverlo de "
            f"{fila['ZONA_ACTUAL']} a {fila['ZONA_RECOMENDADA']}, "
            f"con un ahorro estimado de "
            f"{fila['AHORRO_ESTIMADO_MIN']:.2f} min en la muestra."
        )

    return (
        f"El SKU pertenece al perfil ML '{perfil}' y obtuvo "
        f"un Score de Prioridad de {fila['SCORE_PRIORIDAD']:.2f}. "
        f"El optimizador recomienda mantenerlo en "
        f"{fila['ZONA_ACTUAL']} dentro de las restricciones actuales."
    )

resultado_hibrido["JUSTIFICACION_HIBRIDA"] = (
    resultado_hibrido.apply(
        generar_justificacion_hibrida,
        axis=1
    )
)

resultado_hibrido[
    [
        "SKU",
        "CLUSTER_ML",
        "PERFIL_ML",
        "SCORE_PRIORIDAD",
        "ZONA_ACTUAL",
        "ZONA_RECOMENDADA",
        "MOVIMIENTO",
        "AHORRO_ESTIMADO_MIN",
        "JUSTIFICACION_HIBRIDA"
    ]
].head(20)

# Fase 26 — Visualización del modelo ML

## 26.1 PCA para visualizar los clusters en 2 dimensiones

K-Means utiliza varias variables al mismo tiempo, por lo que no podemos visualizar directamente todos los ejes.

Aplicaremos **PCA** únicamente como herramienta de visualización para proyectar los datos a dos dimensiones.

PCA no cambia el modelo K-Means ya entrenado.

In [ ]:
pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_ml_scaled)

visualizacion_ml = pd.DataFrame({
    "SKU": base_maestra["SKU"].values,
    "PCA_1": X_pca[:, 0],
    "PCA_2": X_pca[:, 1],
    "CLUSTER_ML": base_maestra["CLUSTER_ML"].values,
    "PERFIL_ML": base_maestra["PERFIL_ML"].values
})

varianza_pca = pca.explained_variance_ratio_.sum() * 100

print(
    f"Varianza explicada por los dos componentes: "
    f"{varianza_pca:.2f}%"
)

In [ ]:
plt.figure(figsize=(9, 6))

for cluster in sorted(visualizacion_ml["CLUSTER_ML"].unique()):
    datos_cluster = visualizacion_ml[
        visualizacion_ml["CLUSTER_ML"] == cluster
    ]

    plt.scatter(
        datos_cluster["PCA_1"],
        datos_cluster["PCA_2"],
        label=f"Cluster {cluster}",
        alpha=0.75
    )

plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.title("Visualización PCA de los clusters de SKU")
plt.legend()
plt.tight_layout()
plt.show()

## 26.2 Score promedio por cluster

Este gráfico ayuda a comparar la segmentación aprendida por K-Means con el Score multicriterio.

In [ ]:
score_cluster = (
    recomendaciones
    .groupby(["CLUSTER_ML", "PERFIL_ML"])["SCORE_PRIORIDAD"]
    .mean()
    .reset_index()
    .sort_values("SCORE_PRIORIDAD")
)

etiquetas_score = (
    score_cluster["CLUSTER_ML"].astype(str)
    + " - "
    + score_cluster["PERFIL_ML"]
)

plt.figure(figsize=(8, 5))
plt.barh(
    etiquetas_score,
    score_cluster["SCORE_PRIORIDAD"]
)
plt.xlabel("Score promedio")
plt.ylabel("Cluster")
plt.title("Score de Prioridad promedio por cluster")
plt.tight_layout()
plt.show()

## 26.3 Tasa de movimiento por cluster

Muestra qué proporción de cada perfil fue seleccionada por el optimizador para cambiar de zona.

In [ ]:
mov_cluster = analisis_cluster_optimizacion.sort_values(
    "TASA_MOVIMIENTO_%"
)

etiquetas_mov = (
    mov_cluster["CLUSTER_ML"].astype(str)
    + " - "
    + mov_cluster["PERFIL_ML"]
)

plt.figure(figsize=(8, 5))
plt.barh(
    etiquetas_mov,
    mov_cluster["TASA_MOVIMIENTO_%"]
)
plt.xlabel("SKU movidos dentro del cluster (%)")
plt.ylabel("Cluster")
plt.title("Tasa de movimiento recomendada por cluster")
plt.tight_layout()
plt.show()

# Fase 27 — Exportación del modelo ML y resultados finales

## 27.1 Guardar el modelo entrenado

Guardaremos en un archivo `.joblib`:

- Modelo K-Means.
- StandardScaler.
- Lista de variables utilizadas.
- Número de clusters.
- Métricas principales.

Esto permite reutilizar posteriormente el modelo para asignar nuevos SKU a los perfiles aprendidos, siempre que se utilicen las mismas variables y el mismo criterio de preparación.

In [ ]:
PAQUETE_MODELO_ML = "modelo_kmeans_reslotting_inchcape.joblib"

paquete_modelo = {
    "modelo_kmeans": modelo_kmeans,
    "scaler": scaler,
    "variables_ml": variables_ml,
    "mejor_k": MEJOR_K,
    "silhouette_score": silhouette_final,
    "calinski_harabasz": calinski_final,
    "davies_bouldin": davies_final,
    "perfil_clusters": perfil_clusters_final
}

joblib.dump(
    paquete_modelo,
    PAQUETE_MODELO_ML
)

print("✅ Modelo ML guardado:", PAQUETE_MODELO_ML)

## 27.2 Exportar un Excel final con Machine Learning

El archivo final integra resultados analíticos, ML y optimización.

In [ ]:
ARCHIVO_SALIDA_ML = "resultado_reslotting_inchcape_ML.xlsx"

with pd.ExcelWriter(
    ARCHIVO_SALIDA_ML,
    engine="openpyxl"
) as writer:

    base_maestra.to_excel(
        writer,
        sheet_name="BASE_ML",
        index=False
    )

    evaluacion_k.to_excel(
        writer,
        sheet_name="EVALUACION_K",
        index=False
    )

    metricas_ml.to_excel(
        writer,
        sheet_name="METRICAS_ML",
        index=False
    )

    perfil_clusters_final.to_excel(
        writer,
        sheet_name="PERFILES_CLUSTER",
        index=False
    )

    analisis_cluster_optimizacion.to_excel(
        writer,
        sheet_name="CLUSTER_OPTIMIZACION",
        index=False
    )

    resultado_hibrido.to_excel(
        writer,
        sheet_name="RESULTADO_HIBRIDO",
        index=False
    )

    validacion_capacidad.to_excel(
        writer,
        sheet_name="CAPACIDAD_FINAL",
        index=False
    )

    resumen_kpi.to_excel(
        writer,
        sheet_name="RESUMEN_KPI",
        index=False
    )

print("✅ Archivo final generado:", ARCHIVO_SALIDA_ML)

## 27.3 Resumen final del modelo

Esta celda genera los principales resultados para exposición.

In [ ]:
print("=" * 70)
print("MODELO FINAL — ML + SCORING + OPTIMIZACIÓN")
print("=" * 70)

print(f"SKU analizados: {len(base_maestra)}")
print(f"Variables utilizadas por K-Means: {len(variables_ml)}")
print(f"Número óptimo de clusters seleccionado: {MEJOR_K}")
print(f"Silhouette Score: {silhouette_final:.4f}")
print(f"Calinski-Harabasz: {calinski_final:.4f}")
print(f"Davies-Bouldin: {davies_final:.4f}")
print()
print(f"SKU recomendados para movimiento: {cantidad_movimientos}")
print(f"Ahorro total estimado: {ahorro_total:.2f} min")
print(f"Reducción estimada: {ahorro_porcentaje:.2f}%")

print("\nPerfiles aprendidos:")
print(
    perfil_clusters_final[
        [
            "CLUSTER_ML",
            "CANTIDAD_SKU",
            "PRIORIDAD_CLUSTER_RANK",
            "PERFIL_ML"
        ]
    ]
    .sort_values("PRIORIDAD_CLUSTER_RANK")
    .to_string(index=False)
)

print(
    "\n✅ El componente ML segmenta los SKU; "
    "el optimizador determina la zona factible recomendada."
)

## 27.4 Descargar resultados

Se descargarán:

1. El Excel final con resultados.
2. El archivo `.joblib` con el modelo K-Means entrenado.

In [ ]:
from google.colab import files

files.download(ARCHIVO_SALIDA_ML)
files.download(PAQUETE_MODELO_ML)

# Conclusión metodológica

El sistema desarrollado combina tres componentes:

### 1. Machine Learning no supervisado — K-Means
Descubre perfiles de SKU a partir de su comportamiento físico y operativo.

### 2. Score multicriterio
Genera una priorización interpretable utilizando ahorro, rotación, ABC y facilidad relativa de movimiento.

### 3. Optimización matemática — PuLP
Busca la asignación de zonas que minimiza el tiempo estimado de picking respetando las restricciones disponibles.

## Alcance actual

La salida debe interpretarse como:

> **La mejor decisión factible con las variables y restricciones disponibles en el dataset actual.**

## Evolución futura

Cuando exista información histórica con fechas, el siguiente paso será entrenar un modelo supervisado de forecast para predecir demanda o frecuencia futura de picks.

Esa predicción podrá alimentar al optimizador y convertir el MVP en un sistema de **reslotting dinámico y predictivo**.